In [1]:
!pip install -q -U trl torchao bert-score git+https://github.com/bckim92/language-evaluation.git


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
import os, gc, re, json, time, math, random, zipfile, tarfile, subprocess, threading, importlib
from pathlib import Path
from collections import Counter

import gdown
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from datasets import Dataset, Image as HFImage
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration
from peft import LoraConfig, PeftModel
from trl import DPOConfig, DPOTrainer
from trl.trainer.dpo_trainer import DataCollatorForVisionPreference

SEED = 10623
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# assert torch.cuda.is_available() and "A100" in torch.cuda.get_device_name(0), "Requires A100 GPU."
NUM_EPOCHS = 1
MICRO_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 5e-6
DPO_BETA = 0.1

PROJECT_DIR = Path("/content/drive/MyDrive/EMMbedding")
DATA_DIR = Path("/content/data/artcap")

# Change these two lines when switching between E3 and E2
EXP_NAME = "E3_ranked_dpo"
PAIRS_PATH = PROJECT_DIR / "judge_pairs.jsonl"  # Use PROJECT_DIR / "preferences/random_pairs.jsonl" for E2
RUN_TAG = f"{EXP_NAME}_lr{LEARNING_RATE}_beta{DPO_BETA}_ep{NUM_EPOCHS}"
# Example output: "E3_ranked_dpo_lr5e-06_beta0.1_ep1"

EXP_DIR = PROJECT_DIR / EXP_NAME / RUN_TAG
ADAPTER_DIR = EXP_DIR / "adapter"
PREDICTIONS_DIR = EXP_DIR / "predictions"
for d in [DATA_DIR, EXP_DIR, ADAPTER_DIR, PREDICTIONS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
PROMPT = """Describe the visual content of this painting in one concise, informative sentence.

Focus on:
- The people, objects, and subjects depicted.
- Their actions and spatial relationships.
- Important colors, visual attributes, and setting.

Only describe details that are visually supported by the image.
Do not speculate about the artist, historical context, or symbolic meaning.

Return only the caption."""

In [5]:
def ensure_images_extracted():
    img_dir = DATA_DIR / "image_archive"
    if not img_dir.exists() or not any(img_dir.iterdir()):
        archive = DATA_DIR / "artcap_images_archive"
        if not archive.exists():
            gdown.download("https://drive.google.com/uc?id=1inQ4W-CtNhhfo1vAehSbLgyMPTmltWVa", str(archive), quiet=False, fuzzy=True)
        img_dir.mkdir(parents=True, exist_ok=True)
        if zipfile.is_zipfile(archive):
            with zipfile.ZipFile(archive) as z: z.extractall(img_dir)
        else:
            with tarfile.open(archive) as t: t.extractall(img_dir)
    return {p.stem: p for p in img_dir.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"}}

image_lookup = ensure_images_extracted()
CAPTIONS_JSONL = PROJECT_DIR / "captions.jsonl"

if CAPTIONS_JSONL.exists():
    all_rows = [json.loads(line) for line in open(CAPTIONS_JSONL) if line.strip()]
    # Re-resolve local /content image paths in case Colab runtime restarted
    for r in all_rows:
        stem = str(r["image_id"])
        r["image_path"] = str(image_lookup.get(stem) or image_lookup[str(int(stem))])
else:
    kale_archive = DATA_DIR / "kale_data.zip"
    kale_dir = DATA_DIR / "kale"
    if not kale_dir.exists():
        gdown.download("https://drive.google.com/uc?id=1OdPZMQQACgy2OmRsIp3-W-KnD7ATC-hO", str(kale_archive), quiet=False, fuzzy=True)
        with zipfile.ZipFile(kale_archive) as z: z.extractall(kale_dir)

    def load_split_dict(filename):
        raw = json.load(open(next(kale_dir.rglob(filename))))
        return {Path(str(img_id)).stem: [str(c) for c in caps if c] for img_id, caps in raw}

    tv_ann = load_split_dict("artcap_train_val.json")
    test_ann = load_split_dict("artcap_test.json")
    train_ids, val_ids = train_test_split(list(tv_ann.keys()), test_size=0.1, random_state=SEED)
    train_ids, val_ids = set(train_ids), set(val_ids)

    def build_rows(ann_subset, split):
        rows = []
        for img_id, refs in ann_subset.items():
            path = image_lookup.get(img_id) or image_lookup[str(int(img_id))]
            rows.append({"image_id": str(img_id), "image_path": str(path), "split": split, "references": refs})
        return rows

    all_rows = (
        build_rows({k: tv_ann[k] for k in train_ids}, "train")
        + build_rows({k: tv_ann[k] for k in val_ids}, "val")
        + build_rows(test_ann, "test")
    )
    with open(CAPTIONS_JSONL, "w") as f:
        for r in all_rows: f.write(json.dumps(r) + "\n")

train_rows = [r for r in all_rows if r["split"] == "train"]
val_rows   = [r for r in all_rows if r["split"] == "val"]
test_rows  = [r for r in all_rows if r["split"] == "test"]
print(f"Paintings -> Train: {len(train_rows)}, Val: {len(val_rows)}, Test: {len(test_rows)}")

Downloading...
From (original): https://drive.google.com/uc?id=1inQ4W-CtNhhfo1vAehSbLgyMPTmltWVa
From (redirected): https://drive.google.com/uc?id=1inQ4W-CtNhhfo1vAehSbLgyMPTmltWVa&confirm=t&uuid=d87d4cf7-f2a8-4ada-b917-61b6f410fa3c
To: /content/data/artcap/artcap_images_archive
100%|██████████| 597M/597M [00:05<00:00, 103MB/s] 


Paintings -> Train: 2271, Val: 253, Test: 1082


In [7]:
path_by_id = {str(r["image_id"]): r["image_path"] for r in all_rows}
raw_pairs = [json.loads(line) for line in open(PAIRS_PATH) if line.strip()]

# Filter out any accidental ties and attach current local image_path
clean_pairs = []
for row in raw_pairs:
    if row["chosen"].strip() == row["rejected"].strip():
        continue
    row["image_path"] = path_by_id[str(row["image_id"])]
    clean_pairs.append(row)

train_pairs = [r for r in clean_pairs if r["split"] == "train"]
val_pairs   = [r for r in clean_pairs if r["split"] == "val"]

def to_hf_dataset(pairs):
    ds = Dataset.from_list([
        {
            "image": {"path": r["image_path"], "bytes": None},
            "prompt": [{"role": "user", "content": PROMPT}],
            "chosen": [{"role": "assistant", "content": r["chosen"]}],
            "rejected": [{"role": "assistant", "content": r["rejected"]}],
            "image_id": str(r["image_id"]),
        }
        for r in pairs
    ])
    return ds.cast_column("image", HFImage())

train_dataset = to_hf_dataset(train_pairs)
val_dataset   = to_hf_dataset(val_pairs)
print(f"Loaded {EXP_NAME} pairs -> Train: {len(train_dataset)}, Val: {len(val_dataset)}")

Loaded E3_ranked_dpo pairs -> Train: 4532, Val: 505


In [8]:
def init_model_and_processor(lora_r=16, lora_alpha=32, lora_dropout=0.05):
    processor = AutoProcessor.from_pretrained(
        MODEL_ID,
        size={"shortest_edge": 224 * 28 * 28, "longest_edge": 512 * 28 * 28},
    )
    processor.tokenizer.padding_side = "left"
    if processor.tokenizer.pad_token is None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token

    model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True,
    )
    model.config.use_cache = False

    for name, param in model.named_parameters():
        if "visual" in name.lower():
            param.requires_grad = False

    lora_config = LoraConfig(
        r=lora_r,
        lora_alpha=lora_alpha,
        lora_dropout=lora_dropout,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=r".*language_model.*(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)$",
    )
    return model, processor, lora_config

In [10]:
import os, gc, json, time, math, shutil, urllib.request, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from peft import PeftModel
from trl import DPOConfig, DPOTrainer
from trl.trainer.dpo_trainer import DataCollatorForVisionPreference
from PIL import Image
from tqdm.auto import tqdm

from bert_score import score as bertscore_score
from language_evaluation.coco_caption_py3.pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
from language_evaluation.coco_caption_py3.pycocoevalcap.bleu.bleu import Bleu
from language_evaluation.coco_caption_py3.pycocoevalcap.rouge.rouge import Rouge
from language_evaluation.coco_caption_py3.pycocoevalcap.cider.cider import Cider
import language_evaluation.coco_caption_py3.pycocoevalcap.meteor.meteor as meteor_mod

# ============================================================
# 1. CANDIDATE HYPERPARAMETERS (3 TARGETED VARIATIONS)
# ============================================================
SWEEP_CONFIGS = [
    {"learning_rate": 1e-5, "beta": 0.1,  "num_epochs": 1},  # Stronger policy shift
    {"learning_rate": 5e-6, "beta": 0.05, "num_epochs": 1},  # Weaker KL anchor
    {"learning_rate": 5e-6, "beta": 0.1,  "num_epochs": 2},  # 2-epoch schedule
]

MICRO_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4  # Keeps effective_batch_size = 8

# Ensure METEOR paraphrase data exists in fresh runtime
meteor_dir = os.path.dirname(meteor_mod.__file__)
data_dir = os.path.join(meteor_dir, "data")
os.makedirs(data_dir, exist_ok=True)
paraphrase_path = os.path.join(data_dir, "paraphrase-en.gz")
if not os.path.exists(paraphrase_path) or os.path.getsize(paraphrase_path) == 0:
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/tylin/coco-caption/master/pycocoevalcap/meteor/data/paraphrase-en.gz",
        paraphrase_path,
    )

def compute_meteor_clean(gts, res):
    jar_path = os.path.join(meteor_dir, "meteor-1.5.jar")
    proc = subprocess.Popen(
        ["java", "-Xmx2G", "-XX:-UseContainerSupport", "-Xlog:all=off", "-jar", jar_path, "-", "-", "-stdio", "-l", "en", "-norm"],
        cwd=meteor_dir,
        stdin=subprocess.PIPE,
        stdout=subprocess.PIPE,
        stderr=subprocess.DEVNULL,
    )
    imgIds = list(gts.keys())
    eval_line = "EVAL"
    for i in imgIds:
        hyp = res[i][0].replace("|||", "").replace("  ", " ")
        refs_str = " ||| ".join(gts[i])
        proc.stdin.write(f"SCORE ||| {refs_str} ||| {hyp}\n".encode("utf-8"))
        proc.stdin.flush()
        stat = proc.stdout.readline().decode("utf-8").strip()
        eval_line += f" ||| {stat}"

    proc.stdin.write(f"{eval_line}\n".encode("utf-8"))
    proc.stdin.flush()
    _ = [proc.stdout.readline().decode("utf-8").strip() for _ in imgIds]
    agg_line = proc.stdout.readline().decode("utf-8").strip()

    if " " in agg_line:
        proc.stdin.write(f"SCORE ||| {agg_line}\n".encode("utf-8"))
        proc.stdin.flush()
        score = float(proc.stdout.readline().decode("utf-8").strip())
    else:
        score = float(agg_line)

    proc.kill()
    return score

def eval_adapter_split(adapter_dir, run_dir, rows, split_name="val", batch_size=16, full_metrics=False):
    metrics_path = run_dir / f"{split_name}_metrics.json"
    if metrics_path.exists():
        with open(metrics_path) as f:
            return json.load(f)

    pred_dir = run_dir / "predictions"
    pred_dir.mkdir(parents=True, exist_ok=True)
    pred_csv = pred_dir / f"{split_name}_predictions.csv"

    predictions = pd.read_csv(pred_csv).to_dict("records") if pred_csv.exists() else []
    done_ids = {str(r["image_id"]) for r in predictions}
    remaining = [r for r in rows if str(r["image_id"]) not in done_ids]

    if remaining:
        base_model, processor, _ = init_model_and_processor()
        inf_model = PeftModel.from_pretrained(base_model, str(adapter_dir)).to("cuda").eval()
        inf_model.config.use_cache = True

        for i in tqdm(range(0, len(remaining), batch_size), desc=f"Generating ({split_name})"):
            batch = remaining[i : i + batch_size]
            msgs = [
                [{"role": "user", "content": [
                    {"type": "image", "image": Image.open(r["image_path"]).convert("RGB")},
                    {"type": "text", "text": PROMPT},
                ]}]
                for r in batch
            ]
            inputs = processor.apply_chat_template(
                msgs, tokenize=True, add_generation_prompt=True, padding=True, return_dict=True, return_tensors="pt"
            ).to("cuda")

            with torch.inference_mode():
                out_ids = inf_model.generate(**inputs, max_new_tokens=128, do_sample=False)

            trimmed = [o[len(inp):] for inp, o in zip(inputs["input_ids"], out_ids)]
            captions = [
                t.strip()
                for t in processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)
            ]
            for r, cap in zip(batch, captions):
                predictions.append({
                    "image_id": str(r["image_id"]),
                    "prediction": cap,
                    "references": json.dumps(r["references"]),
                })
            pd.DataFrame(predictions).to_csv(pred_csv, index=False)

        del inf_model, base_model
        gc.collect()
        torch.cuda.empty_cache()

    df = pd.read_csv(pred_csv)
    preds = df["prediction"].astype(str).tolist()
    refs = [json.loads(x) for x in df["references"]]

    tokenizer = PTBTokenizer()
    gts = tokenizer.tokenize({i: [{"caption": c} for c in r] for i, r in enumerate(refs)})
    res = tokenizer.tokenize({i: [{"caption": p}] for i, p in enumerate(preds)})

    results = {}
    cider_score, _ = Cider().compute_score(gts, res)
    results["CIDEr"] = float(cider_score)
    results["avg_pred_words"] = float(np.mean([len(p.split()) for p in preds]))

    if full_metrics:
        bleu_scores, _ = Bleu(4).compute_score(gts, res)
        for idx, b in enumerate(bleu_scores, 1):
            results[f"Bleu_{idx}"] = float(b)

        rouge_score, _ = Rouge().compute_score(gts, res)
        results["ROUGE_L"] = float(rouge_score)

        bert_P, bert_R, bert_F1 = bertscore_score(
            cands=preds, refs=refs, lang="en", model_type="roberta-large",
            device="cuda", batch_size=16, verbose=False, rescale_with_baseline=False,
        )
        results["BERTScore_P"] = float(bert_P.mean().item())
        results["BERTScore_R"] = float(bert_R.mean().item())
        results["BERTScore_F1"] = float(bert_F1.mean().item())
        results["METEOR"] = float(compute_meteor_clean(gts, res))

    with open(metrics_path, "w") as f:
        json.dump(results, f, indent=2)
    return results

# ============================================================
# 2. RUN VALIDATION SWEEP ACROSS CANDIDATES
# ============================================================
EXP_ROOT = PROJECT_DIR / "E3_ranked_dpo"

for cfg in SWEEP_CONFIGS:
    lr, beta, ep = cfg["learning_rate"], cfg["beta"], cfg["num_epochs"]
    run_tag = f"E3_ranked_dpo_lr{lr}_beta{beta}_ep{ep}"
    run_dir = EXP_ROOT / run_tag
    adapter_dir = run_dir / "adapter"
    ckpt_dir = run_dir / "checkpoints"
    run_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n{'=' * 65}\nRUN: {run_tag}\n{'=' * 65}")

    # Skip training if adapter already saved on Drive
    if not (adapter_dir / "adapter_config.json").exists():
        effective_bs = MICRO_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
        est_steps = math.ceil(len(train_dataset) / effective_bs) * ep

        model, processor, lora_config = init_model_and_processor()
        vision_collator = DataCollatorForVisionPreference(processor=processor, max_length=None)

        training_args = DPOConfig(
            output_dir=str(ckpt_dir),
            num_train_epochs=ep,
            per_device_train_batch_size=MICRO_BATCH_SIZE,
            gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
            learning_rate=lr,
            beta=beta,
            loss_type="sigmoid",
            lr_scheduler_type="cosine",
            warmup_steps=int(0.05 * est_steps),
            optim="adamw_torch",
            max_grad_norm=0.3,
            bf16=True,
            tf32=True,
            gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False},
            max_length=None,
            logging_steps=10,
            save_strategy="no",  # Save final adapter only to save Drive space
            remove_unused_columns=False,
            dataloader_num_workers=2,
            dataloader_pin_memory=True,
            report_to="none",
            seed=SEED,
            data_seed=SEED,
        )

        trainer = DPOTrainer(
            model=model,
            ref_model=None,
            args=training_args,
            train_dataset=train_dataset,
            processing_class=processor,
            peft_config=lora_config,
            data_collator=vision_collator,
        )

        torch.cuda.empty_cache()
        train_result = trainer.train()
        trainer.save_model(str(adapter_dir))
        processor.save_pretrained(str(adapter_dir))

        with open(run_dir / "experiment_config.json", "w") as f:
            json.dump({**cfg, "run_tag": run_tag, "seed": SEED}, f, indent=2)
        with open(run_dir / "training_metrics.json", "w") as f:
            json.dump(train_result.metrics, f, indent=2, default=str)

        del trainer, model
        gc.collect()
        torch.cuda.empty_cache()

    val_res = eval_adapter_split(adapter_dir, run_dir, val_rows, split_name="val", full_metrics=False)
    print(f"Val CIDEr (raw / x100): {val_res['CIDEr']:.4f} / {val_res['CIDEr']*100:.2f} | Avg Words: {val_res['avg_pred_words']:.1f}")

# ============================================================
# 3. RANK ALL E3 RUNS (INCLUDING INITIAL BASELINE) BY VAL CIDEr
# ============================================================
leaderboard = []
for run_path in sorted(EXP_ROOT.iterdir()):
    if not run_path.is_dir():
        continue
    # Support both baseline filename and sweep filename
    m_file = run_path / "val_metrics.json"
    if not m_file.exists():
        m_file = run_path / "E3_ranked_dpo_val_metrics.json"
    if not m_file.exists():
        continue

    data = json.load(open(m_file))
    metrics_dict = data.get("metrics", data)
    leaderboard.append({
        "run_tag": run_path.name,
        "val_CIDEr_raw": metrics_dict["CIDEr"],
        "val_CIDEr_x100": metrics_dict["CIDEr"] * 100,
        "avg_pred_words": data.get("avg_pred_words", metrics_dict.get("avg_pred_words")),
        "adapter_dir": str(run_path / "adapter"),
        "run_dir": str(run_path),
    })

lb_df = pd.DataFrame(leaderboard).sort_values("val_CIDEr_raw", ascending=False).reset_index(drop=True)
display(lb_df[["run_tag", "val_CIDEr_raw", "val_CIDEr_x100", "avg_pred_words"]])

# ============================================================
# 4. RUN FULL 10-METRIC TEST EVALUATION ON THE WINNING MODEL
# ============================================================
winner = lb_df.iloc[0]
print(f"\nWinner selected by Validation CIDEr: {winner['run_tag']} ({winner['val_CIDEr_x100']:.2f})")

final_test_metrics = eval_adapter_split(
    Path(winner["adapter_dir"]),
    Path(winner["run_dir"]),
    test_rows,
    split_name="test",
    full_metrics=True,
)

pct_metrics = {"Bleu_1", "Bleu_2", "Bleu_3", "Bleu_4", "METEOR", "ROUGE_L", "CIDEr", "BERTScore_P", "BERTScore_R", "BERTScore_F1"}
final_df = pd.DataFrame([
    {
        "Metric": k,
        "Raw Score": v,
        "Reported Score": v * 100 if k in pct_metrics else v,
    }
    for k, v in final_test_metrics.items()
])
final_df.to_csv(Path(winner["run_dir"]) / "final_winner_test_metrics.csv", index=False)
display(final_df.style.format({"Raw Score": "{:.4f}", "Reported Score": "{:.4f}"}))


RUN: E3_ranked_dpo_lr1e-05_beta0.1_ep1


Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

Step,Training Loss
10,0.692465
20,0.694231
30,0.657920
40,0.619561
50,0.515557
60,0.490934
70,0.507753
80,0.426805
90,0.455329
100,0.420932


Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

Generating (val):   0%|          | 0/16 [00:00<?, ?it/s]

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transform

... (25 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


,Metric,Raw Score,Reported Score
0,CIDEr,0.0224,2.2401
1,avg_pred_words,19.2338,19.2338
2,Bleu_1,0.3247,32.4689
3,Bleu_2,0.1575,15.7489
4,Bleu_3,0.0690,6.8987
5,Bleu_4,0.0350,3.4963
6,ROUGE_L,0.2563,25.6279
7,BERTScore_P,0.8982,89.8250
8,BERTScore_R,0.9236,92.3556
9,BERTScore_F1,0.9083,90.8269


In [11]:
import os, json, urllib.request, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from bert_score import score as bertscore_score
from language_evaluation.coco_caption_py3.pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
from language_evaluation.coco_caption_py3.pycocoevalcap.bleu.bleu import Bleu
from language_evaluation.coco_caption_py3.pycocoevalcap.rouge.rouge import Rouge
from language_evaluation.coco_caption_py3.pycocoevalcap.cider.cider import Cider
import language_evaluation.coco_caption_py3.pycocoevalcap.meteor.meteor as meteor_mod

EXP_ROOT = Path("/content/drive/MyDrive/EMMbedding/E3_ranked_dpo")
device = "cuda" if torch.cuda.is_available() else "cpu"

# Ensure METEOR paraphrase data exists
meteor_dir = os.path.dirname(meteor_mod.__file__)
data_dir = os.path.join(meteor_dir, "data")
os.makedirs(data_dir, exist_ok=True)
paraphrase_path = os.path.join(data_dir, "paraphrase-en.gz")
if not os.path.exists(paraphrase_path) or os.path.getsize(paraphrase_path) == 0:
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/tylin/coco-caption/master/pycocoevalcap/meteor/data/paraphrase-en.gz",
        paraphrase_path,
    )

def compute_meteor_clean(gts, res):
    jar_path = os.path.join(meteor_dir, "meteor-1.5.jar")
    proc = subprocess.Popen(
        ["java", "-Xmx2G", "-XX:-UseContainerSupport", "-Xlog:all=off", "-jar", jar_path, "-", "-", "-stdio", "-l", "en", "-norm"],
        cwd=meteor_dir,
        stdin=subprocess.PIPE,
        stdout=subprocess.PIPE,
        stderr=subprocess.DEVNULL,
    )
    imgIds = list(gts.keys())
    eval_line = "EVAL"
    for i in imgIds:
        hyp = res[i][0].replace("|||", "").replace("  ", " ")
        refs_str = " ||| ".join(gts[i])
        proc.stdin.write(f"SCORE ||| {refs_str} ||| {hyp}\n".encode("utf-8"))
        proc.stdin.flush()
        stat = proc.stdout.readline().decode("utf-8").strip()
        eval_line += f" ||| {stat}"

    proc.stdin.write(f"{eval_line}\n".encode("utf-8"))
    proc.stdin.flush()
    _ = [proc.stdout.readline().decode("utf-8").strip() for _ in imgIds]
    agg_line = proc.stdout.readline().decode("utf-8").strip()

    if " " in agg_line:
        proc.stdin.write(f"SCORE ||| {agg_line}\n".encode("utf-8"))
        proc.stdin.flush()
        score = float(proc.stdout.readline().decode("utf-8").strip())
    else:
        score = float(agg_line)

    proc.kill()
    return score

summary_rows = []
for run_dir in sorted(EXP_ROOT.iterdir()):
    if not run_dir.is_dir():
        continue
    val_csv = run_dir / "predictions" / "val_predictions.csv"
    if not val_csv.exists():
        val_csv = run_dir / "predictions" / "E3_ranked_dpo_val_predictions.csv"
    if not val_csv.exists():
        continue

    df = pd.read_csv(val_csv)
    preds = df["prediction"].astype(str).tolist()
    refs = [json.loads(x) for x in df["references"]]

    tokenizer = PTBTokenizer()
    gts = tokenizer.tokenize({i: [{"caption": c} for c in r] for i, r in enumerate(refs)})
    res = tokenizer.tokenize({i: [{"caption": p}] for i, p in enumerate(preds)})

    cider_s, _ = Cider().compute_score(gts, res)
    bleu_s, _ = Bleu(4).compute_score(gts, res)
    rouge_s, _ = Rouge().compute_score(gts, res)
    meteor_s = compute_meteor_clean(gts, res)
    bert_P, bert_R, bert_F1 = bertscore_score(
        cands=preds, refs=refs, lang="en", model_type="roberta-large",
        device=device, batch_size=16, verbose=False, rescale_with_baseline=False,
    )

    full_res = {
        "CIDEr": float(cider_s),
        "Bleu_1": float(bleu_s[0]),
        "Bleu_2": float(bleu_s[1]),
        "Bleu_3": float(bleu_s[2]),
        "Bleu_4": float(bleu_s[3]),
        "METEOR": float(meteor_s),
        "ROUGE_L": float(rouge_s),
        "BERTScore_P": float(bert_P.mean().item()),
        "BERTScore_R": float(bert_R.mean().item()),
        "BERTScore_F1": float(bert_F1.mean().item()),
        "avg_pred_words": float(np.mean([len(p.split()) for p in preds])),
    }

    with open(run_dir / "val_metrics_full.json", "w") as f:
        json.dump(full_res, f, indent=2)

    summary_rows.append({
        "Run": run_dir.name,
        "C": round(full_res["CIDEr"] * 100, 4),
        "B1": round(full_res["Bleu_1"] * 100, 4),
        "B2": round(full_res["Bleu_2"] * 100, 4),
        "B3": round(full_res["Bleu_3"] * 100, 4),
        "B4": round(full_res["Bleu_4"] * 100, 4),
        "M": round(full_res["METEOR"] * 100, 4),
        "R": round(full_res["ROUGE_L"] * 100, 4),
        "BERT_F1": round(full_res["BERTScore_F1"] * 100, 4),
        "Words": round(full_res["avg_pred_words"], 1),
    })

full_val_df = pd.DataFrame(summary_rows).sort_values("C", ascending=False).reset_index(drop=True)
full_val_df.to_csv(EXP_ROOT / "all_e3_val_comparison.csv", index=False)
display(full_val_df)

{'testlen': 5641, 'reflen': 3781, 'guess': [5641, 5388, 5135, 4882], 'correct': [1656, 332, 54, 24]}
ratio: 1.4919333509649586


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'testlen': 4866, 'reflen': 3657, 'guess': [4866, 4613, 4360, 4107], 'correct': [1566, 335, 63, 27]}
ratio: 1.3305988515172735


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


... (3 outputs left out)

{'testlen': 5314, 'reflen': 3762, 'guess': [5314, 5061, 4808, 4555], 'correct': [1570, 320, 58, 26]}
ratio: 1.4125465178093002


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


,Run,C,B1,B2,B3,B4,M,R,BERT_F1,Words
0,E3_ranked_dpo_lr5e-06_beta0.05_ep1,3.2300,32.1825,15.2876,6.9638,3.8601,11.9242,25.2962,90.7166,19.2
1,E3_ranked_dpo_lr5e-06_beta0.1_ep2,2.6450,29.5446,13.6677,6.0853,3.3677,11.6699,23.9110,90.0816,21.0
2,E3_ranked_dpo_lr5e-06_beta0.1_ep1,2.6263,29.3559,13.5428,6.1184,3.2983,11.6680,23.7870,89.8886,22.9
3,E3_ranked_dpo_lr1e-05_beta0.1_ep1,2.5352,29.3565,13.4495,5.7512,3.1097,11.6329,23.5314,89.9459,22.3


##Single Run

In [6]:
effective_batch_size = MICRO_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
estimated_steps = math.ceil(len(train_dataset) / effective_batch_size) * NUM_EPOCHS

model, processor, lora_config = init_model_and_processor()
vision_collator = DataCollatorForVisionPreference(processor=processor, max_length=None)

training_args = DPOConfig(
    output_dir=str(EXP_DIR / "checkpoints"),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=MICRO_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    beta=DPO_BETA,
    loss_type="sigmoid",
    lr_scheduler_type="cosine",
    warmup_steps=int(0.05 * estimated_steps),
    optim="adamw_torch",
    max_grad_norm=0.3,
    bf16=True,
    tf32=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_length=None,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=1,
    remove_unused_columns=False,
    dataloader_num_workers=2,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
)

trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=processor,
    peft_config=lora_config,
    data_collator=vision_collator,
)

torch.cuda.empty_cache()
train_result = trainer.train()

trainer.save_model(str(ADAPTER_DIR))
processor.save_pretrained(str(ADAPTER_DIR))

with open(EXP_DIR / "training_metrics.json", "w") as f:
    json.dump(train_result.metrics, f, indent=2, default=str)

del trainer, model
gc.collect()
torch.cuda.empty_cache()
print("Saved adapter to:", ADAPTER_DIR)

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/5.50k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.50k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

... (5 outputs left out)

Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

Step,Training Loss
10,0.693677
20,0.688003
30,0.685911
40,0.671584
50,0.634141
60,0.611787
70,0.576179
80,0.520199
90,0.516401
100,0.488786


Saved adapter to: /content/drive/MyDrive/EMMbedding/E3_ranked_dpo/E3_ranked_dpo_lr5e-06_beta0.1_ep1/adapter


In [11]:
# !pip install -q bert-score git+https://github.com/bckim92/language-evaluation.git

# from google.colab import drive
# drive.mount("/content/drive")

import os
import json
import time
import subprocess
import threading
import importlib
import torch
import pandas as pd
import numpy as np

from pathlib import Path
from bert_score import score as bertscore_score

from language_evaluation.coco_caption_py3.pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
from language_evaluation.coco_caption_py3.pycocoevalcap.bleu.bleu import Bleu
from language_evaluation.coco_caption_py3.pycocoevalcap.rouge.rouge import Rouge
from language_evaluation.coco_caption_py3.pycocoevalcap.cider.cider import Cider
import language_evaluation.coco_caption_py3.pycocoevalcap.meteor.meteor as meteor_mod

importlib.reload(meteor_mod)
from language_evaluation.coco_caption_py3.pycocoevalcap.meteor.meteor import Meteor

# ============================================================
# 1. LOAD SAVED E3 PREDICTIONS & REFERENCES FROM DRIVE
# ============================================================

MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
EXP_DIR = Path("/content/drive/MyDrive/EMMbedding/E3_ranked_dpo/E3_ranked_dpo_lr5e-06_beta0.1_ep1")
PRED_CSV = EXP_DIR / "predictions" / "E3_ranked_dpo_test_predictions.csv"

predictions_df = pd.read_csv(PRED_CSV)
predictions = predictions_df["prediction"].astype(str).tolist()
answers = [json.loads(value) for value in predictions_df["references"]]

assert len(predictions) == len(answers)
assert all(len(refs) == 5 for refs in answers)

print("Predictions:", len(predictions))
print("Reference sets:", len(answers))
print("Total references:", sum(len(x) for x in answers))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Evaluation device:", device)

# ============================================================
# 2. CONSTRUCT COCO-STYLE INPUTS & PTB TOKENIZATION
# ============================================================

gts_raw = {
    i: [{"caption": caption} for caption in references]
    for i, references in enumerate(answers)
}
res_raw = {
    i: [{"caption": prediction}]
    for i, prediction in enumerate(predictions)
}

print("\nStarting PTB tokenization...", flush=True)
start = time.time()

tokenizer = PTBTokenizer()
gts = tokenizer.tokenize(gts_raw)
res = tokenizer.tokenize(res_raw)

print(f"Tokenization completed in {time.time() - start:.2f}s")

# ============================================================
# 3. SETUP RESULTS AND CHECKPOINTING
# ============================================================

evaluation_results = {}
metric_times = {}

RESULT_PATH = EXP_DIR / "E3_ranked_dpo_test_metrics.json"


def save_checkpoint():
    checkpoint = {
        "model": MODEL_ID,
        "experiment": "E3 - Ranked DPO (lr=5e-6, beta=0.1, ep=1)",
        "num_images": len(predictions),
        "num_references_per_image": 5,
        "avg_pred_words": float(np.mean([len(p.split()) for p in predictions])),
        "metrics": evaluation_results,
        "metric_times_seconds": metric_times,
    }
    with open(RESULT_PATH, "w") as f:
        json.dump(checkpoint, f, indent=2)


# ============================================================
# 4. RUN BLEU, ROUGE, CIDEr FIRST (FAST & PURE PYTHON)
# ============================================================

fast_scorers = [
    ("BLEU", Bleu(4), ["Bleu_1", "Bleu_2", "Bleu_3", "Bleu_4"]),
    ("ROUGE", Rouge(), "ROUGE_L"),
    ("CIDEr", Cider(), "CIDEr"),
]

for name, scorer, metric_names in fast_scorers:
    print(f"\n{'=' * 55}")
    print(f"Starting {name} evaluation...")
    print(f"{'=' * 55}", flush=True)

    start = time.time()
    score, individual_scores = scorer.compute_score(gts, res)
    elapsed = time.time() - start
    metric_times[name] = elapsed

    if isinstance(metric_names, list):
        for metric, value in zip(metric_names, score):
            evaluation_results[metric] = float(value)
            print(f"{metric}: {value * 100:.4f}", flush=True)
    else:
        evaluation_results[metric_names] = float(score)
        print(f"{metric_names}: {score * 100:.4f}", flush=True)

    print(f"{name} runtime: {elapsed / 60:.2f} minutes")
    save_checkpoint()
    print("Progress saved!", flush=True)

# ============================================================
# 5. BERTSCORE (RUN BEFORE METEOR SO IT CHECKPOINTS SAFELY)
# ============================================================

print("\n" + "=" * 55)
print("Starting BERTScore evaluation...")
print("=" * 55, flush=True)

start = time.time()

bert_P, bert_R, bert_F1 = bertscore_score(
    cands=predictions,
    refs=answers,
    lang="en",
    model_type="roberta-large",
    device=str(device),
    batch_size=16,
    verbose=True,
    rescale_with_baseline=False,
)

bert_precision = bert_P.mean().item()
bert_recall = bert_R.mean().item()
bert_f1 = bert_F1.mean().item()

evaluation_results["BERTScore_P"] = bert_precision
evaluation_results["BERTScore_R"] = bert_recall
evaluation_results["BERTScore_F1"] = bert_f1

metric_times["BERTScore"] = time.time() - start

print(f"BERTScore Precision: {bert_precision * 100:.4f}")
print(f"BERTScore Recall:    {bert_recall * 100:.4f}")
print(f"BERTScore F1:        {bert_f1 * 100:.4f}")
print(f"BERTScore runtime: {metric_times['BERTScore'] / 60:.2f} minutes")

bert_df = pd.DataFrame({
    "image_id": predictions_df["image_id"].astype(str),
    "BERTScore_P": bert_P.cpu().numpy(),
    "BERTScore_R": bert_R.cpu().numpy(),
    "BERTScore_F1": bert_F1.cpu().numpy(),
})

bert_df.to_csv(EXP_DIR / "E3_ranked_dpo_bertscore.csv", index=False)
save_checkpoint()
print("BERTScore results saved!")

# ============================================================
# 6. METEOR LAST (HANDLES TWO-STEP STATS -> SCORE PROTOCOL)
# ============================================================

print("\n" + "=" * 55)
print("Starting METEOR evaluation...")
print("=" * 55, flush=True)


def compute_meteor_safe(gts, res):
    meteor_scorer = Meteor()
    imgIds = list(gts.keys())
    scores = []

    eval_line = "EVAL"
    meteor_scorer.lock.acquire()
    try:
        for i in imgIds:
            assert len(res[i]) == 1
            stat = meteor_scorer._stat(res[i][0], gts[i])
            eval_line += f" ||| {stat}"

        meteor_scorer.meteor_p.stdin.write(f"{eval_line}\n".encode("utf-8"))
        meteor_scorer.meteor_p.stdin.flush()

        raw_lines = [
            meteor_scorer.meteor_p.stdout.readline().decode("utf-8").strip()
            for _ in imgIds
        ]
        # Read the trailing aggregate line emitted by EVAL
        agg_line = meteor_scorer.meteor_p.stdout.readline().decode("utf-8").strip()

        # Check if meteor-1.5.jar returned sufficient stats instead of direct floats
        if " " in raw_lines[0]:
            for stats_str in raw_lines:
                meteor_scorer.meteor_p.stdin.write(f"SCORE ||| {stats_str}\n".encode("utf-8"))
                meteor_scorer.meteor_p.stdin.flush()
                scores.append(float(meteor_scorer.meteor_p.stdout.readline().decode("utf-8").strip()))

            meteor_scorer.meteor_p.stdin.write(f"SCORE ||| {agg_line}\n".encode("utf-8"))
            meteor_scorer.meteor_p.stdin.flush()
            score = float(meteor_scorer.meteor_p.stdout.readline().decode("utf-8").strip())
        else:
            scores = [float(x) for x in raw_lines]
            score = float(agg_line)
    finally:
        meteor_scorer.lock.release()
        try:
            meteor_scorer.meteor_p.kill()
        except Exception:
            pass

    return score, scores


start = time.time()
meteor_score, _ = compute_meteor_safe(gts, res)
elapsed = time.time() - start
metric_times["METEOR"] = elapsed
evaluation_results["METEOR"] = float(meteor_score)

print(f"METEOR: {meteor_score * 100:.4f}", flush=True)
print(f"METEOR runtime: {elapsed / 60:.2f} minutes")
save_checkpoint()
print("Progress saved!", flush=True)

# ============================================================
# 7. DISPLAY & SAVE FINAL RESULTS (IDENTICAL FORMAT TO E1 & E2)
# ============================================================

print("\n" + "=" * 65)
print("FINAL E3 (RANKED DPO - BASELINE HYPERPARAMS) RESULTS")
print("=" * 65)

percentage_metrics = {
    "Bleu_1",
    "Bleu_2",
    "Bleu_3",
    "Bleu_4",
    "METEOR",
    "ROUGE_L",
    "CIDEr",
    "BERTScore_P",
    "BERTScore_R",
    "BERTScore_F1",
}

metrics_df = pd.DataFrame([
    {
        "Metric": metric,
        "Raw Score": score,
        "Reported Score": score * 100 if metric in percentage_metrics else score,
        "Scale": "x100" if metric in percentage_metrics else "Standard CLIPScore",
    }
    for metric, score in evaluation_results.items()
])

display(
    metrics_df.style.format({
        "Raw Score": "{:.4f}",
        "Reported Score": "{:.4f}",
    })
)

metrics_df.to_csv(EXP_DIR / "E3_ranked_dpo_test_metrics.csv", index=False)
save_checkpoint()

print("\nAll metrics completed!")
print("Results saved to:", EXP_DIR)

Predictions: 1082
Reference sets: 1082
Total references: 5410
Evaluation device: cuda

Starting PTB tokenization...
Tokenization completed in 0.53s

Starting BLEU evaluation...
{'testlen': 24262, 'reflen': 16372, 'guess': [24262, 23180, 22098, 21016], 'correct': [7076, 1512, 231, 72]}
ratio: 1.4819203518200903
Bleu_1: 29.1649
Bleu_2: 13.7927
Bleu_3: 5.8370
Bleu_4: 2.8730
BLEU runtime: 0.01 minutes
Progress saved!

Starting ROUGE evaluation...
ROUGE_L: 24.3643
ROUGE runtime: 0.00 minutes
Progress saved!

Starting CIDEr evaluation...
CIDEr: 1.5256
CIDEr runtime: 0.02 minutes
Progress saved!

Starting BERTScore evaluation...


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


calculating scores...
computing bert embedding.


  0%|          | 0/404 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/339 [00:00<?, ?it/s]

done in 9.65 seconds, 560.65 sentences/sec
BERTScore Precision: 88.7524
BERTScore Recall:    92.1178
BERTScore F1:        90.1659
BERTScore runtime: 0.21 minutes
BERTScore results saved!

Starting METEOR evaluation...


BrokenPipeError: [Errno 32] Broken pipe

In [13]:
import os, json, time, urllib.request, subprocess
from pathlib import Path
import pandas as pd
import language_evaluation.coco_caption_py3.pycocoevalcap.meteor.meteor as meteor_mod
from language_evaluation.coco_caption_py3.pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer

EXP_DIR = Path("/content/drive/MyDrive/EMMbedding/E3_ranked_dpo/E3_ranked_dpo_lr5e-06_beta0.1_ep1")
RESULT_PATH = EXP_DIR / "E3_ranked_dpo_test_metrics.json"
PRED_CSV = EXP_DIR / "predictions" / "E3_ranked_dpo_test_predictions.csv"

# 1. Download missing paraphrase-en.gz into meteor_dir/data/
meteor_dir = os.path.dirname(meteor_mod.__file__)
data_dir = os.path.join(meteor_dir, "data")
os.makedirs(data_dir, exist_ok=True)
paraphrase_path = os.path.join(data_dir, "paraphrase-en.gz")

if not os.path.exists(paraphrase_path) or os.path.getsize(paraphrase_path) == 0:
    print("Downloading paraphrase-en.gz...")
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/tylin/coco-caption/master/pycocoevalcap/meteor/data/paraphrase-en.gz",
        paraphrase_path,
    )
    print("Downloaded:", paraphrase_path, os.path.getsize(paraphrase_path), "bytes")

# 2. Load saved predictions & tokenize
predictions_df = pd.read_csv(PRED_CSV)
predictions = predictions_df["prediction"].astype(str).tolist()
answers = [json.loads(v) for v in predictions_df["references"]]

tokenizer = PTBTokenizer()
gts = tokenizer.tokenize({i: [{"caption": c} for c in r] for i, r in enumerate(answers)})
res = tokenizer.tokenize({i: [{"caption": p}] for i, p in enumerate(predictions)})

# 3. Run METEOR with clean JVM flags and two-step EVAL -> SCORE protocol
def compute_meteor_clean(gts, res):
    jar_path = os.path.join(meteor_dir, "meteor-1.5.jar")
    proc = subprocess.Popen(
        ["java", "-Xmx2G", "-XX:-UseContainerSupport", "-Xlog:all=off", "-jar", jar_path, "-", "-", "-stdio", "-l", "en", "-norm"],
        cwd=meteor_dir,
        stdin=subprocess.PIPE,
        stdout=subprocess.PIPE,
        stderr=subprocess.DEVNULL,
    )
    imgIds = list(gts.keys())
    eval_line = "EVAL"
    for i in imgIds:
        hyp = res[i][0].replace("|||", "").replace("  ", " ")
        refs_str = " ||| ".join(gts[i])
        proc.stdin.write(f"SCORE ||| {refs_str} ||| {hyp}\n".encode("utf-8"))
        proc.stdin.flush()
        stat = proc.stdout.readline().decode("utf-8").strip()
        eval_line += f" ||| {stat}"

    proc.stdin.write(f"{eval_line}\n".encode("utf-8"))
    proc.stdin.flush()

    raw_lines = [proc.stdout.readline().decode("utf-8").strip() for _ in imgIds]
    agg_line = proc.stdout.readline().decode("utf-8").strip()

    if " " in agg_line:
        proc.stdin.write(f"SCORE ||| {agg_line}\n".encode("utf-8"))
        proc.stdin.flush()
        score = float(proc.stdout.readline().decode("utf-8").strip())
    else:
        score = float(agg_line)

    proc.kill()
    return score

start = time.time()
meteor_score = compute_meteor_clean(gts, res)
elapsed = time.time() - start
print(f"METEOR: {meteor_score * 100:.4f} (in {elapsed:.2f}s)")

# 4. Update saved checkpoint JSON & display full E3 table
with open(RESULT_PATH, "r") as f:
    checkpoint = json.load(f)

checkpoint["metrics"]["METEOR"] = float(meteor_score)
checkpoint["metric_times_seconds"]["METEOR"] = elapsed

with open(RESULT_PATH, "w") as f:
    json.dump(checkpoint, f, indent=2)

percentage_metrics = {
    "Bleu_1", "Bleu_2", "Bleu_3", "Bleu_4",
    "METEOR", "ROUGE_L", "CIDEr",
    "BERTScore_P", "BERTScore_R", "BERTScore_F1",
}
metrics_df = pd.DataFrame([
    {
        "Metric": m,
        "Raw Score": s,
        "Reported Score": s * 100 if m in percentage_metrics else s,
    }
    for m, s in checkpoint["metrics"].items()
])
metrics_df.to_csv(EXP_DIR / "E3_ranked_dpo_test_metrics.csv", index=False)
display(metrics_df.style.format({"Raw Score": "{:.4f}", "Reported Score": "{:.4f}"}))

Downloaded: /usr/local/lib/python3.13/dist-packages/language_evaluation/coco_caption_py3/pycocoevalcap/meteor/data/paraphrase-en.gz 61813011 bytes
METEOR: 11.4783 (in 11.13s)


,Metric,Raw Score,Reported Score
0,Bleu_1,0.2916,29.1649
1,Bleu_2,0.1379,13.7927
2,Bleu_3,0.0584,5.8370
3,Bleu_4,0.0287,2.8730
4,ROUGE_L,0.2436,24.3643
5,CIDEr,0.0153,1.5256
6,BERTScore_P,0.8875,88.7524
7,BERTScore_R,0.9212,92.1178
8,BERTScore_F1,0.9017,90.1659
9,METEOR,0.1148,11.4783


In [14]:
# !apt-get update -qq && apt-get install -y -qq openjdk-11-jre-headless
# !pip install -q bert-score git+https://github.com/bckim92/language-evaluation.git

# from google.colab import drive
# drive.mount("/content/drive")

import os
import json
import time
import urllib.request
import subprocess
import torch
import pandas as pd
import numpy as np

from pathlib import Path
from bert_score import score as bertscore_score

from language_evaluation.coco_caption_py3.pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
from language_evaluation.coco_caption_py3.pycocoevalcap.bleu.bleu import Bleu
from language_evaluation.coco_caption_py3.pycocoevalcap.rouge.rouge import Rouge
from language_evaluation.coco_caption_py3.pycocoevalcap.cider.cider import Cider
import language_evaluation.coco_caption_py3.pycocoevalcap.meteor.meteor as meteor_mod

# ============================================================
# 1. LOAD SAVED VALIDATION PREDICTIONS FROM DRIVE
# ============================================================

MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
EXP_DIR = Path("/content/drive/MyDrive/EMMbedding/E3_ranked_dpo/E3_ranked_dpo_lr5e-06_beta0.1_ep1")
VAL_PRED_CSV = EXP_DIR / "predictions" / "E3_ranked_dpo_val_predictions.csv"

predictions_df = pd.read_csv(VAL_PRED_CSV)
predictions = predictions_df["prediction"].astype(str).tolist()
answers = [json.loads(value) for value in predictions_df["references"]]

assert len(predictions) == len(answers)
print("Validation predictions:", len(predictions))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Evaluation device:", device)

# ============================================================
# 2. PTB TOKENIZATION
# ============================================================

gts_raw = {
    i: [{"caption": caption} for caption in references]
    for i, references in enumerate(answers)
}
res_raw = {
    i: [{"caption": prediction}]
    for i, prediction in enumerate(predictions)
}

tokenizer = PTBTokenizer()
gts = tokenizer.tokenize(gts_raw)
res = tokenizer.tokenize(res_raw)

# ============================================================
# 3. SETUP RESULTS & CHECKPOINTING
# ============================================================

evaluation_results = {}
metric_times = {}
RESULT_PATH = EXP_DIR / "E3_ranked_dpo_val_metrics.json"


def save_checkpoint():
    checkpoint = {
        "model": MODEL_ID,
        "split": "val",
        "experiment": "E3 - Ranked DPO (lr=5e-6, beta=0.1, ep=1)",
        "num_images": len(predictions),
        "num_references_per_image": 5,
        "avg_pred_words": float(np.mean([len(p.split()) for p in predictions])),
        "metrics": evaluation_results,
        "metric_times_seconds": metric_times,
    }
    with open(RESULT_PATH, "w") as f:
        json.dump(checkpoint, f, indent=2)


# ============================================================
# 4. BLEU, ROUGE, CIDEr
# ============================================================

fast_scorers = [
    ("BLEU", Bleu(4), ["Bleu_1", "Bleu_2", "Bleu_3", "Bleu_4"]),
    ("ROUGE", Rouge(), "ROUGE_L"),
    ("CIDEr", Cider(), "CIDEr"),
]

for name, scorer, metric_names in fast_scorers:
    start = time.time()
    score, _ = scorer.compute_score(gts, res)
    metric_times[name] = time.time() - start

    if isinstance(metric_names, list):
        for metric, value in zip(metric_names, score):
            evaluation_results[metric] = float(value)
    else:
        evaluation_results[metric_names] = float(score)

save_checkpoint()
print("BLEU, ROUGE, and CIDEr saved.")

# ============================================================
# 5. BERTSCORE (EXACT E1/E2 PARITY)
# ============================================================

start = time.time()
bert_P, bert_R, bert_F1 = bertscore_score(
    cands=predictions,
    refs=answers,
    lang="en",
    model_type="roberta-large",
    device=str(device),
    batch_size=16,
    verbose=False,
    rescale_with_baseline=False,
)

evaluation_results["BERTScore_P"] = bert_P.mean().item()
evaluation_results["BERTScore_R"] = bert_R.mean().item()
evaluation_results["BERTScore_F1"] = bert_F1.mean().item()
metric_times["BERTScore"] = time.time() - start

pd.DataFrame({
    "image_id": predictions_df["image_id"].astype(str),
    "BERTScore_P": bert_P.cpu().numpy(),
    "BERTScore_R": bert_R.cpu().numpy(),
    "BERTScore_F1": bert_F1.cpu().numpy(),
}).to_csv(EXP_DIR / "E3_ranked_dpo_val_bertscore.csv", index=False)

save_checkpoint()
print("BERTScore saved.")

# ============================================================
# 6. METEOR (AUTO-DOWNLOADS paraphrase-en.gz + CLEAN JVM)
# ============================================================

meteor_dir = os.path.dirname(meteor_mod.__file__)
data_dir = os.path.join(meteor_dir, "data")
os.makedirs(data_dir, exist_ok=True)
paraphrase_path = os.path.join(data_dir, "paraphrase-en.gz")

if not os.path.exists(paraphrase_path) or os.path.getsize(paraphrase_path) == 0:
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/tylin/coco-caption/master/pycocoevalcap/meteor/data/paraphrase-en.gz",
        paraphrase_path,
    )


def compute_meteor_clean(gts, res):
    jar_path = os.path.join(meteor_dir, "meteor-1.5.jar")
    proc = subprocess.Popen(
        ["java", "-Xmx2G", "-XX:-UseContainerSupport", "-Xlog:all=off", "-jar", jar_path, "-", "-", "-stdio", "-l", "en", "-norm"],
        cwd=meteor_dir,
        stdin=subprocess.PIPE,
        stdout=subprocess.PIPE,
        stderr=subprocess.DEVNULL,
    )
    imgIds = list(gts.keys())
    eval_line = "EVAL"
    for i in imgIds:
        hyp = res[i][0].replace("|||", "").replace("  ", " ")
        refs_str = " ||| ".join(gts[i])
        proc.stdin.write(f"SCORE ||| {refs_str} ||| {hyp}\n".encode("utf-8"))
        proc.stdin.flush()
        stat = proc.stdout.readline().decode("utf-8").strip()
        eval_line += f" ||| {stat}"

    proc.stdin.write(f"{eval_line}\n".encode("utf-8"))
    proc.stdin.flush()

    _ = [proc.stdout.readline().decode("utf-8").strip() for _ in imgIds]
    agg_line = proc.stdout.readline().decode("utf-8").strip()

    if " " in agg_line:
        proc.stdin.write(f"SCORE ||| {agg_line}\n".encode("utf-8"))
        proc.stdin.flush()
        score = float(proc.stdout.readline().decode("utf-8").strip())
    else:
        score = float(agg_line)

    proc.kill()
    return score


start = time.time()
meteor_score = compute_meteor_clean(gts, res)
metric_times["METEOR"] = time.time() - start
evaluation_results["METEOR"] = float(meteor_score)

save_checkpoint()

# ============================================================
# 7. DISPLAY & EXPORT FINAL VALIDATION TABLE
# ============================================================

percentage_metrics = {
    "Bleu_1", "Bleu_2", "Bleu_3", "Bleu_4",
    "METEOR", "ROUGE_L", "CIDEr",
    "BERTScore_P", "BERTScore_R", "BERTScore_F1",
}

metrics_df = pd.DataFrame([
    {
        "Metric": m,
        "Raw Score": s,
        "Reported Score": s * 100 if m in percentage_metrics else s,
    }
    for m, s in evaluation_results.items()
])

metrics_df.to_csv(EXP_DIR / "E3_ranked_dpo_val_metrics.csv", index=False)
display(metrics_df.style.format({"Raw Score": "{:.4f}", "Reported Score": "{:.4f}"}))

Validation predictions: 253
Evaluation device: cuda
{'testlen': 5791, 'reflen': 3820, 'guess': [5791, 5538, 5285, 5032], 'correct': [1699, 345, 65, 25]}
ratio: 1.5159685863870378
BLEU, ROUGE, and CIDEr saved.


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore saved.


,Metric,Raw Score,Reported Score
0,Bleu_1,0.2934,29.3386
1,Bleu_2,0.1352,13.5193
2,Bleu_3,0.0608,6.0803
3,Bleu_4,0.0325,3.2508
4,ROUGE_L,0.2377,23.7651
5,CIDEr,0.0255,2.5483
6,BERTScore_P,0.8833,88.3257
7,BERTScore_R,0.9197,91.9678
8,BERTScore_F1,0.8989,89.8884
9,METEOR,0.1164,11.6408
